# Screen synthetic accounts with client KumoRelational

Which accounts will receive a simulated suspicious-activity report in the next 30 days? Use pretrained client KumoRelational to evaluate held-out accounts, then score newly added accounts with historical transfer relationships.

Run locally with Python 3.12 or newer and access to an existing KumoRelational NIM endpoint. The five synthetic tables illustrate the workflow; evaluation metrics are **demonstration only**. See [relational requests and predictive queries](https://github.com/NVIDIA/structured-data-models-client/blob/main/docs/get-started/quickstart.md#several-related-tables) for the client interface.

## Install and configure

Install the package into this notebook kernel.
Use an existing NIM endpoint serving the requested model; set `SDM_NIM_URL` and, when required, `SDM_API_KEY` in the environment.
No endpoint is provisioned by this notebook.

In [ ]:
%pip install "structured-data-models-client[relational]" pandas pyarrow scikit-learn

## Set the example parameters

Keep the common cutoff and separate context/holdout populations fixed before evaluation. The endpoint controls pretrained weights; the sampling settings below control the local relational request.

In [ ]:
# Synthetic data and the frozen observation date.
SEED = 42
N_ACCOUNTS = 240
N_TRANSFERS = 2400
N_NEW_ACCOUNTS = 40
N_NEW_TRANSFERS = 400
CUTOFF = "2026-10-01"
HISTORY_DAYS = 180
HORIZON_DAYS = 30
REPORT_BASE_RATE = 0.04
REPORT_SIGNAL = 0.65
SPLIT_FRACTIONS = (0.60, 0.15, 0.10, 0.15)

# Scoring and output choices.
THRESHOLD = 0.5
OUTPUT_PARENT = None  # None uses the system temporary directory.
CLEANUP = False       # Keep saved predictions after the run.

# Pretrained inference and neighborhood sampling.
NUM_CONTEXT_SAMPLES = 144
NUM_NEIGHBORS = [10, 10]
NUM_HOPS = 2


Import the client and create a new local directory for the batch results.

In [ ]:
import os
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
from sdm_client import KumoRelational, PredictiveQuery, Relationship, Schema, SDMClient

workspace = tempfile.TemporaryDirectory(prefix="money-muling-relational-", dir=OUTPUT_PARENT, delete=False)
run_dir = Path(workspace.name)
print("Results:", run_dir)


## Generate synthetic example data

Run the next cell to create historical accounts, transfers, postings, and future report labels, plus a separate account network reserved for scoring. You can skim this cell; the client workflow starts at **Prepare labeled accounts and relationships**.
Historical events precede the common cutoff; future reports supply labels only and never become graph features.

In [ ]:
def make_report_network(n_accounts, n_transfers, prefix, seed):
    local_rng = np.random.default_rng(seed)
    ids = np.array([f"synthetic-{prefix}-account-{i:04d}" for i in range(n_accounts)])
    propensity = local_rng.beta(1.2, 5.0, n_accounts)
    activity_weights = (0.5 + 8 * propensity) / (0.5 + 8 * propensity).sum()
    account_rows = pd.DataFrame({
        "account_id": ids,
        "account_age_days": local_rng.integers(HISTORY_DAYS + 1, 2400, n_accounts),
        "account_type": local_rng.choice(["checking", "savings"], n_accounts, p=[0.8, 0.2]),
        "synthetic_region": local_rng.choice(["region_a", "region_b", "region_c"], n_accounts),
        "mule_label": np.zeros(n_accounts, dtype=np.int64),
    })
    senders = local_rng.choice(n_accounts, n_transfers, p=activity_weights)
    receivers = local_rng.choice(n_accounts, n_transfers, p=activity_weights)
    receivers = np.where(receivers == senders, (receivers + 1) % n_accounts, receivers)
    times = cutoff - pd.to_timedelta(local_rng.integers(1, HISTORY_DAYS * 86400, n_transfers), unit="s")
    transfer_rows = pd.DataFrame({
        "transaction_id": [f"synthetic-{prefix}-transfer-{i:05d}" for i in range(n_transfers)],
        "timestamp": times, "amount": np.round(local_rng.lognormal(4.2, 1.0, n_transfers), 2),
        "channel": local_rng.choice(["online", "mobile", "branch"], n_transfers, p=[0.45, 0.5, 0.05]),
    })
    result = {"accounts": account_rows, "transactions": transfer_rows}
    for name, key, positions in (("inputs", "input_id", senders), ("outputs", "output_id", receivers)):
        result[name] = pd.DataFrame({
            key: [f"synthetic-{prefix}-{name}-{i:05d}" for i in range(n_transfers)],
            "transaction_id": transfer_rows.transaction_id,
            "account_id": ids[positions], "timestamp": times,
        })
    flagged = local_rng.random(n_accounts) < REPORT_BASE_RATE + REPORT_SIGNAL * propensity
    report_rows = pd.DataFrame({
        "report_id": [f"synthetic-{prefix}-report-{i:04d}" for i in range(flagged.sum())],
        "account_id": ids[flagged],
        "timestamp": cutoff + pd.to_timedelta(
            local_rng.integers(1, HORIZON_DAYS * 86400 + 1, flagged.sum()), unit="s"
        ),
    })
    return result, report_rows

cutoff = pd.Timestamp(CUTOFF)
historical_frames, reports = make_report_network(N_ACCOUNTS, N_TRANSFERS, "history", SEED)
new_frames, _ = make_report_network(N_NEW_ACCOUNTS, N_NEW_TRANSFERS, "new", SEED + 1)
accounts, transactions = historical_frames["accounts"], historical_frames["transactions"]
inputs, outputs = historical_frames["inputs"], historical_frames["outputs"]
account_ids = accounts.account_id.to_numpy()
frames = {**historical_frames, "reports": reports}
rng = np.random.default_rng(SEED + 2)
display(pd.DataFrame({"historical_rows": {name: len(frame) for name, frame in historical_frames.items()},
                      "reserved_rows": {name: len(frame) for name, frame in new_frames.items()}}))


Check the keys and time boundaries before modelling. All feature events must precede the cutoff, and reports must fall within the future label window.


In [ ]:
primary_keys = {"accounts": "account_id", "transactions": "transaction_id",
                "inputs": "input_id", "outputs": "output_id", "reports": "report_id"}
for collection in (historical_frames, new_frames):
    for name, frame in collection.items():
        assert frame[primary_keys[name]].notna().all() and frame[primary_keys[name]].is_unique
    for name in ("inputs", "outputs"):
        postings = collection[name]
        assert postings.account_id.isin(collection["accounts"].account_id).all()
        assert postings.transaction_id.is_unique
        pd.testing.assert_frame_equal(postings[["transaction_id", "timestamp"]],
                                      collection["transactions"][["transaction_id", "timestamp"]])
    assert collection["inputs"].account_id.ne(collection["outputs"].account_id).all()
    assert collection["transactions"].timestamp.lt(cutoff).all()
    assert np.isfinite(collection["transactions"].amount).all() and collection["transactions"].amount.gt(0).all()
    assert collection["accounts"].account_age_days.gt(HISTORY_DAYS).all()
    assert collection["accounts"].mule_label.eq(0).all()
for name, frame in new_frames.items():
    assert set(frame[primary_keys[name]]).isdisjoint(historical_frames[name][primary_keys[name]])
    assert frame.dtypes.equals(historical_frames[name].dtypes)
assert reports.report_id.is_unique and reports.account_id.isin(accounts.account_id).all()
assert reports.timestamp.gt(cutoff).all()
assert reports.timestamp.le(cutoff + pd.Timedelta(days=HORIZON_DAYS)).all()
print("Keys, historical features, reserved IDs, and future label windows are valid.")


## Prepare labeled accounts and relationships

Randomly partition the accounts before evaluation. With the defaults, 144 accounts supply labeled context, 36 form the final holdout, and 60 are excluded from both groups. The pretrained model is not tuned on these records. Final holdout labels remain outside the request, although historical records may connect accounts across groups.


In [ ]:
labels = pd.DataFrame({"ENTITY1": account_ids, "TARGET1": np.isin(account_ids, reports.account_id)})
account_order = rng.permutation(N_ACCOUNTS)
split_boundaries = np.rint(np.cumsum(SPLIT_FRACTIONS) * N_ACCOUNTS).astype(int)
split = np.empty(N_ACCOUNTS, dtype=np.int64)
for split_id, positions in enumerate(np.split(account_order, split_boundaries[:-1])):
    split[positions] = split_id
labels["SPLIT"] = split
trainval = labels.loc[labels.SPLIT != 3].copy()
heldout = labels.loc[labels.SPLIT == 3].copy()
requests = heldout[["ENTITY1"]].copy()
assert set(trainval.ENTITY1).isdisjoint(heldout.ENTITY1)
assert requests.ENTITY1.is_unique and "TARGET1" not in requests
assert labels.groupby("SPLIT").TARGET1.nunique().eq(2).all()
split_summary = labels.groupby("SPLIT").TARGET1.agg(accounts="size", report_rate="mean")
split_summary.index = ["train/context", "validation", "internal test", "final holdout"]
display(split_summary)


Keep true labels only on context accounts and mark their eligibility explicitly. Masking unknown labels alone is insufficient for a comparison query. The schema links historical accounts, transfers and postings; report records never become model features.


In [ ]:
context_ids = set(labels.loc[labels.SPLIT == 0, "ENTITY1"])
assert context_ids.isdisjoint(heldout.ENTITY1)
model_accounts = accounts.copy()
model_accounts["mule_label"] = np.where(labels.SPLIT == 0, labels.TARGET1.astype(int), np.nan)
model_accounts["context_eligible"] = labels.SPLIT.eq(0).astype(int)
assert model_accounts.loc[~model_accounts.account_id.isin(context_ids), "mule_label"].isna().all()

def build_schema(tables):
    result = Schema.from_data(
        tables,
        primary_keys={name: primary_keys[name] for name in tables},
        time_columns={"accounts": None, "transactions": "timestamp", "inputs": "timestamp", "outputs": "timestamp"},
        relationships=[Relationship(posting, key, destination, key)
                       for posting in ("inputs", "outputs")
                       for key, destination in (("account_id", "accounts"), ("transaction_id", "transactions"))],
    )
    result.check_consistent()
    return result

context_frames = {**historical_frames, "accounts": model_accounts}
schema = build_schema(context_frames)
print(f"Historical schema ready; {len(context_ids)} accounts have eligible context labels.")


## Ask the account-screening question

The comparison predicts a positive report label in the frozen cohort. `WHERE` selects labelled context; explicit `indices` select the accounts to score. The materializer may print a warning explaining this distinction. Exclude the eligibility marker from model features.


In [ ]:
QUERY = "PREDICT accounts.mule_label = 1 FOR EACH accounts.account_id WHERE accounts.context_eligible = 1"
query = PredictiveQuery(QUERY, schema)
materialize_options = {
    "num_context_samples": NUM_CONTEXT_SAMPLES, "anchor_time": cutoff,
    "num_neighbors": NUM_NEIGHBORS, "num_hops": NUM_HOPS, "random_seed": SEED,
    "exclude_cols_dict": {"accounts": ["context_eligible"]},
}
request = query.materialize(indices=heldout.ENTITY1.tolist(), model=KumoRelational(), **materialize_options)
print("Task:", request.task.kind)


## Inspect the evaluation request

Check that the materialized context contains only eligible labeled accounts and that every requested holdout account is present without a target column. The reported row count reflects the context accepted by the materializer.


In [ ]:
assert request.task.kind == "binary_classification"
assert 1 < request.context_rows <= len(context_ids)
assert request.predict_rows == len(heldout)
assert request.task.target.column_name not in request.predict.instance_table
print(f"Request: {request.context_rows} labelled examples and {request.predict_rows} accounts to score.")


## Prepare the baseline

Compare returned probabilities with the prevalence of reports in the labeled context.

In [ ]:
baseline_probability = np.full(len(heldout), labels.loc[labels.SPLIT == 0, "TARGET1"].mean())

## Evaluate held-out accounts

Send the label-free holdout population and labeled historical context to the existing KumoRelational NIM. The response contains one result per requested account.

In [ ]:
with SDMClient(os.environ["SDM_NIM_URL"], api_key=os.environ.get("SDM_API_KEY")) as client:
    answer = client.predict(request, model=KumoRelational(), outputs=["prediction", "probabilities"])
raw_predictions = answer.to_pandas()


Join by returned account ID instead of assuming response order. Check that every requested account appears once and that both class probabilities are finite, bounded and sum to one. The boolean comparison’s positive class is named `probability_True`.


In [ ]:
def align_predictions(frame, expected_ids):
    expected_ids = list(expected_ids)
    assert {"id", "row_index", "probability_True", "probability_False"} <= set(frame)
    assert frame.id.is_unique and set(frame.id) == set(expected_ids)
    assert pd.api.types.is_integer_dtype(frame.row_index)
    assert frame.row_index.is_unique and set(frame.row_index) == set(range(len(expected_ids)))
    assert all(row.id == expected_ids[row.row_index] for row in frame.itertuples())
    probabilities = frame[["probability_False", "probability_True"]].to_numpy()
    assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, rtol=1e-5, atol=1e-6)
    return frame.set_index("id").loc[expected_ids].reset_index()

aligned_predictions = align_predictions(raw_predictions, heldout.ENTITY1)
probability = aligned_predictions.probability_True.to_numpy()


Higher AUROC and average precision indicate better ranking within this simulation; lower Brier score means smaller probability error. Compare with the train-context prevalence baseline. Every value is **demonstration only**, and no improvement is required on this small artificial sample.


In [ ]:
def quality(actual, probability):
    return {"AUROC": roc_auc_score(actual, probability),
            "Average precision": average_precision_score(actual, probability),
            "Brier score": brier_score_loss(actual, probability)}

metrics = pd.DataFrame({
    "Prevalence baseline": quality(heldout.TARGET1, baseline_probability),
    "Client Relational": quality(heldout.TARGET1, probability),
}).T
print(f"Final holdout: {len(heldout)} accounts; report rate: {heldout.TARGET1.mean():.1%}.")
display(metrics)


## Batch predictions

Append the reserved accounts and their historical relationships to copies of the original tables, then build a fresh schema. New accounts carry unknown labels and cannot enter the labeled context. The original evaluation schema remains unchanged.

In [ ]:
scoring_accounts = new_frames["accounts"].copy()
scoring_accounts["mule_label"] = np.nan
scoring_accounts["context_eligible"] = 0
new_model_frames = {**new_frames, "accounts": scoring_accounts}
scoring_frames = {}
for name, frame in context_frames.items():
    assert set(new_model_frames[name][primary_keys[name]]).isdisjoint(frame[primary_keys[name]])
    assert list(new_model_frames[name].columns) == list(frame.columns)
    assert new_model_frames[name].dtypes.equals(frame.dtypes)
    scoring_frames[name] = pd.concat([frame, new_model_frames[name]], ignore_index=True)
scoring_ids = scoring_accounts.account_id.tolist()
assert set(scoring_ids).isdisjoint(labels.ENTITY1)
assert scoring_frames["accounts"].loc[
    scoring_frames["accounts"].account_id.isin(scoring_ids), "mule_label"
].isna().all()
scoring_schema = build_schema(scoring_frames)
scoring_query = PredictiveQuery(QUERY, scoring_schema)
scoring_request = scoring_query.materialize(indices=scoring_ids, model=KumoRelational(), **materialize_options)
assert scoring_request.predict_rows == len(scoring_ids)
assert scoring_request.task.target.column_name not in scoring_request.predict.instance_table
print("New accounts to score:", scoring_request.predict_rows)

Send the new account population with the same eligible historical context to the endpoint.

In [ ]:
with SDMClient(os.environ["SDM_NIM_URL"], api_key=os.environ.get("SDM_API_KEY")) as client:
    batch_answer = client.predict(scoring_request, model=KumoRelational(), outputs=["prediction", "probabilities"])

### Inspect and save new account scores

Expect one row per new account. `report_probability` estimates the simulated report outcome; `review_flag` applies the illustrative threshold. The flag is not a business policy or an allegation. Match results by returned ID and preview a few rows before saving.


In [ ]:
batch_scores = align_predictions(batch_answer.to_pandas(), scoring_ids)
predictions = pd.DataFrame({"account_id": scoring_ids, "report_probability": batch_scores.probability_True.to_numpy()})
predictions["review_flag"] = predictions.report_probability >= THRESHOLD
prediction_path = run_dir / "client_relational_predictions.parquet"
predictions.to_parquet(prediction_path, index=False)
display(predictions.head())
print(f"Saved {len(predictions)} newly added account scores to {prediction_path}.")


## Retain the scoring inputs

Save the combined feature tables with only eligible context labels, along with scoring IDs and the query. Rebuild their schema and request to repeat this batch; the endpoint controls pretrained weights. The prediction file is not a trained-model bundle.

In [ ]:
for name, frame in scoring_frames.items():
    frame.to_parquet(run_dir / f"{name}.parquet", index=False)
pd.DataFrame({"account_id": scoring_ids}).to_parquet(run_dir / "scoring_ids.parquet", index=False)
(run_dir / "query.txt").write_text(QUERY + "\n")
print("Scoring tables, IDs, and query saved in", run_dir)

## Keep or remove this run’s files

The output directory is retained by default. Enable `CLEANUP` to remove only the new directory owned by this execution. The shared model cache and installed environment remain available.


In [ ]:
if CLEANUP:
    workspace.cleanup()
print("Removed this run's files." if CLEANUP else f"Results retained in {run_dir}.")
